In [18]:
#jupyter nbconvert --to python preprocesamiento.ipynb. #Convertir codigo python punta a punta

import pandas as pd
import numpy as np

#LOAD DATASETS
datafiles = {
    "Fenomenos": "../Dataset_INUMET/Fenomenos.csv",
    "Humedad": "../Dataset_INUMET/Humedad.csv",
    "Lluvia": "../Dataset_INUMET/Lluvia.csv",
    "Temperatura": "../Dataset_INUMET/Temperatura.csv",
    "Visibilidad": "../Dataset_INUMET/Visibilidad.csv",
    "VientoSalto": "../Dataset_INUMET/VientoSalto.csv",
    "VientoCarrasco": "../Dataset_INUMET/VientoCarrasco.csv"
}

dataframes = {}
for name, path in datafiles.items():
    dataframes[name] = pd.read_csv(path, sep = ";", encoding='utf-8')
# Access data like: dataframes["Temperature"]


In [19]:
# ====== 1. Lluvia.csv ======
print("Procesando datos Lluvia.csv")
df_lluvia = dataframes["Lluvia"].copy()

df_lluvia = df_lluvia.rename(columns={
    "Estación": "estacion",
    "Cod. Pluvio": "cod_pluvio",
    "Fecha": "fecha",
    "[mm]": "precip_mm",
    "Comentario": "coment_lluv"
})

# Convertir fecha y precipitación
df_lluvia["fecha"] = pd.to_datetime(df_lluvia["fecha"], format="%d/%m/%Y", errors="coerce")
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].replace("TRAZA", "0.0")  #! Nota: Transformación TRAZA se traduce a 0.0 
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].astype(float)

#! Criterio: día lluvioso si precipitación >= 1.0 mm
precip_minima = 1.0
df_lluvia["dia_lluvioso"] = (df_lluvia["precip_mm"] >= precip_minima).astype(int)

# Seleccionamos solo columnas útiles para el df_global
df_lluvia = df_lluvia[["fecha", "precip_mm", "dia_lluvioso"]]

print(df_lluvia.head())


Procesando datos Lluvia.csv
       fecha  precip_mm  dia_lluvioso
0 1950-01-01        4.6             1
1 1950-01-02        0.0             0
2 1950-01-03        0.0             0
3 1950-01-04        0.0             0
4 1950-01-05        0.0             0


In [20]:
# ====== 2. Humedad.csv ======
print("Procesando datos Humedad.csv")
df_hum = dataframes["Humedad"].copy()

df_hum = df_hum.rename(columns={
    "Fecha/hora": "fecha",
    "HumedadCarrasco ": "hum_carrasco",  
    "HumedadSalto ": "hum_salto"
})

# Convertir fecha
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"], errors="coerce")
df_hum["fecha"] = df_hum["fecha"].dt.date  # Extraer solo la fecha (sin hora)
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"])  # Volver a datetime

# Eliminar columnas no deseadas
df_hum = df_hum.drop(columns=["Unnamed: 3", "Unnamed: 4"], errors="ignore")  # errors="ignore" evita error si no existen

# Agrupar por fecha (en caso de múltiples mediciones por día)
df_hum = df_hum.groupby("fecha").agg({
    "hum_carrasco": "mean",
    "hum_salto": "mean"
}).reset_index()

print(df_hum.head())
print(df_hum.isna().sum())

Procesando datos Humedad.csv
       fecha  hum_carrasco  hum_salto
0 1980-01-01           NaN       69.0
1 1980-01-02          60.0       61.0
2 1980-01-03          76.0       63.0
3 1980-01-04          60.0       82.0
4 1980-01-05          66.0       84.0
fecha              0
hum_carrasco      12
hum_salto       1425
dtype: int64


In [21]:
# ====== 3. Temperatura.csv ======
print("Procesando datos Temperatura.csv")

df_temp = dataframes["Temperatura"].copy()

df_temp = df_temp.rename(columns={
    "Fecha": "fecha",
    "TmáxCarrasco": "tmax_carrasco",
    "TmínCarrasco": "tmin_carrasco",
    "TmáxSalto": "tmax_salto",
    "TmínSalto": "tmin_salto"
})

df_temp["fecha"] = pd.to_datetime(df_temp["fecha"], errors="coerce", dayfirst=True) 
# Eliminar fechas no válidas
df_temp = df_temp.dropna(subset=["fecha"]) #! existen varias fechas no definidas, nAT 
df_temp = df_temp.drop(columns=["Unnamed: 5"], errors="ignore")  

df_temp["amplitud_carrasco"] = df_temp["tmax_carrasco"] - df_temp["tmin_carrasco"]
df_temp["amplitud_salto"] = df_temp["tmax_salto"] - df_temp["tmin_salto"]
df_temp["tprom_carrasco"] = (df_temp["tmax_carrasco"] + df_temp["tmin_carrasco"]) / 2
df_temp["tprom_salto"] = (df_temp["tmax_salto"] + df_temp["tmin_salto"]) / 2

df_temp


Procesando datos Temperatura.csv


,fecha,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,tprom_salto
0,1970-01-01,32.6,21.0,32.0,19.8,11.6,12.2,26.80,25.90
1,1970-01-02,33.8,22.0,35.4,20.7,11.8,14.7,27.90,28.05
2,1970-01-03,28.1,21.5,30.8,21.3,6.6,9.5,24.80,26.05
3,1970-01-04,24.0,18.8,34.1,20.0,5.2,14.1,21.40,27.05
4,1970-01-05,30.5,20.2,37.2,23.0,10.3,14.2,25.35,30.10
...,...,...,...,...,...,...,...,...,...
20303,2025-08-03,17.3,8.4,20.7,11.3,8.9,9.4,12.85,16.00
20304,2025-08-04,13.9,6.3,17.5,8.2,7.6,9.3,10.10,12.85
20305,2025-08-05,12.0,7.5,17.1,3.4,4.5,13.7,9.75,10.25
20306,2025-08-06,15.5,2.8,18.3,3.4,12.7,14.9,9.15,10.85


In [22]:
# ===== VientoCarrasco.csv =====
df_vientocarr = dataframes["VientoCarrasco"]

df_vientocarr = df_vientocarr.rename(columns={
  "Fecha " : "fecha",
  "DirVCarrasco" : "dir_viento_carrasco",
  "IntVCarrasco" : "int_viento_carrasco",
  "IntRCarrasco" : "int_racha_viento_carrasco"
})

df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"], dayfirst=True)
df_vientocarr["fecha"] = df_vientocarr["fecha"].dt.date
df_vientocarr["fecha"] = pd.to_datetime(df_vientocarr["fecha"])
df_vientocarr["dir_viento_carrasco"] = pd.to_numeric(df_vientocarr["dir_viento_carrasco"],errors="coerce")

df_vientocarr = df_vientocarr.groupby("fecha").agg({
    "dir_viento_carrasco": "mean",                  # promedio
    "int_viento_carrasco": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_carrasco": ["mean", "max"]    # promedio y máximo
})
df_vientocarr.columns = [
    "dir_viento_carrasco_prom",
    "int_viento_carrasco_prom",
    "int_viento_carrasco_max",
    "int_racha_viento_carrasco_prom",
    "int_racha_viento_carrasco_max"
]

df_vientocarr.head()

,dir_viento_carrasco_prom,int_viento_carrasco_prom,int_viento_carrasco_max,int_racha_viento_carrasco_prom,int_racha_viento_carrasco_max
fecha,,,,,
1986-07-07,52.500000,12.687500,20.0,NaN,NaN
1986-07-08,41.666667,15.500000,25.0,NaN,NaN
1986-07-09,93.333333,9.416667,15.0,NaN,NaN
1986-07-10,62.916667,12.291667,19.0,NaN,NaN
1986-07-11,179.166667,14.333333,22.0,NaN,NaN


In [23]:
# ===== VientoSalto.csv =====
df_vientosalto = dataframes["VientoSalto"]

df_vientosalto = df_vientosalto.rename(columns={
  "Fecha/hora" : "fecha",
  "DirVSalto" : "dir_viento_salto",
  "IntVSalto" : "int_viento_salto",
  "IntRSalto" : "int_racha_viento_salto"
})

df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"], dayfirst=True)
df_vientosalto["fecha"] = df_vientosalto["fecha"].dt.date
df_vientosalto["fecha"] = pd.to_datetime(df_vientosalto["fecha"])
df_vientosalto["dir_viento_salto"] = pd.to_numeric(df_vientosalto["dir_viento_salto"],errors="coerce")

df_vientosalto = df_vientosalto.groupby("fecha").agg({
    "dir_viento_salto": "mean",                  # promedio
    "int_viento_salto": ["mean", "max"],         # promedio y máximo
    "int_racha_viento_salto": ["mean", "max"]    # promedio y máximo
})
df_vientosalto.columns = [
    "dir_viento_salto_prom",
    "int_viento_salto_prom",
    "int_viento_salto_max",
    "int_racha_viento_salto_prom",
    "int_racha_viento_salto_max"
]

df_vientosalto.head()

,dir_viento_salto_prom,int_viento_salto_prom,int_viento_salto_max,int_racha_viento_salto_prom,int_racha_viento_salto_max
fecha,,,,,
1980-01-01,151.250000,5.250,9,NaN,NaN
1980-01-02,132.500000,5.875,8,NaN,NaN
1980-01-03,233.750000,4.375,7,NaN,NaN
1980-01-04,71.428571,4.250,6,NaN,NaN
1980-01-05,123.750000,10.250,14,NaN,NaN


In [24]:
# ===== Visibilidad.csv =====

df_vis = dataframes["Visibilidad"]

df_vis = df_vis.rename(columns={
    "Fecha/hora": "fecha",
    "VisibilidadCarrasco ": "vis_carrasco",
    "VisibilidadSalto ": "vis_salto"
})

df_vis["fecha"] = pd.to_datetime(
    df_vis["fecha"],
    dayfirst=True,
    infer_datetime_format=True,  # infiere si hay hora o no
    errors = "coerce"
)
df_vis["fecha"] = df_vis["fecha"].dt.date
df_vis["fecha"] = pd.to_datetime(df_vis["fecha"])

df_vis["vis_carrasco"] = pd.to_numeric(df_vis["vis_carrasco"], errors="coerce")
df_vis["vis_salto"] = pd.to_numeric(df_vis["vis_salto"], errors="coerce")

df_vis = df_vis.groupby("fecha").agg({
    "vis_carrasco": ["mean", "max", "min"],
    "vis_salto": ["mean", "max", "min"]
})

df_vis.columns = [
    "vis_carrasco_prom", "vis_carrasco_max", "vis_carrasco_min",
    "vis_salto_prom", "vis_salto_max", "vis_salto_min"
]

df_vis.head()

/tmp/ipykernel_66718/1862289934.py:11: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_vis["fecha"] = pd.to_datetime(


,vis_carrasco_prom,vis_carrasco_max,vis_carrasco_min,vis_salto_prom,vis_salto_max,vis_salto_min
fecha,,,,,,
1995-07-03,NaN,NaN,NaN,11.260870,15.0,6.0
1995-07-04,NaN,NaN,NaN,12.956522,15.0,10.0
1995-07-05,NaN,NaN,NaN,8.565217,10.0,6.0
1995-07-06,NaN,NaN,NaN,13.347826,20.0,4.0
1995-07-07,NaN,NaN,NaN,7.652174,15.0,2.0


In [25]:
# ===== Fenomenos.csv =====

df_fen = dataframes["Fenomenos"]

df_fen = df_fen.rename(columns={
    "Fecha/hora": "fecha",
    "FenomenosCarrasco ": "fen_carrasco",
    "FenomenosSalto ": "fen_salto"
})
df_fen = df_fen.drop(columns=["Unnamed: 3", "Unnamed: 4"], axis=1)

df_fen["fecha"] = df_fen["fecha"].astype(str).str.split(" ").str[0]
df_fen["fecha"] = pd.to_datetime(df_fen["fecha"], dayfirst=True, errors="coerce")

df_carr = df_fen[["fecha", "fen_carrasco"]].copy()
df_carr["fen_carrasco"] = df_carr["fen_carrasco"].fillna(0).astype(int).astype(str)  # 0 = sin fenómeno
df_carr = pd.get_dummies(df_carr, columns=["fen_carrasco"])

df_salto = df_fen[["fecha", "fen_salto"]].copy()
df_salto["fen_salto"] = df_salto["fen_salto"].fillna(0).astype(int).astype(str)
df_salto = pd.get_dummies(df_salto, columns=["fen_salto"])

df_carr_daily = df_carr.groupby("fecha").sum().clip(upper=1).reset_index()
df_salto_daily = df_salto.groupby("fecha").sum().clip(upper=1).reset_index()

df_fen = pd.merge(df_carr_daily, df_salto_daily, on="fecha", how="outer")

df_fen.head()



,fecha,fen_carrasco_0,fen_carrasco_1,fen_carrasco_10,fen_carrasco_11,fen_carrasco_12,fen_carrasco_13,fen_carrasco_14,fen_carrasco_15,fen_carrasco_16,...,fen_salto_80,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99
0,2000-01-01,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2000-01-02,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2000-01-03,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2000-01-04,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2000-01-05,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
# Merge de dataset procesados

print("Combinando datasets en df_global")

df_global = df_lluvia.copy()
# Hacer merge con humedad
df_global = pd.merge(df_global, df_hum, on="fecha", how="outer")

# Hacer merge con temperatura
df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")

"""
df_global = df_global[[ "fecha",  
                       "hum_carrasco", "hum_salto",
                       "tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto", "dia_lluvioso"]]
"""

df_global = pd.merge(df_global,df_vientocarr,how="left",on="fecha")
df_global = pd.merge(df_global,df_vientosalto,how="left",on="fecha")
df_global = pd.merge(df_global,df_vis,how="left",on="fecha")
df_global = pd.merge(df_global,df_fen,how="left",on="fecha")
df_global = df_global[[c for c in df_global.columns if c != "dia_lluvioso"] + ["dia_lluvioso"]] #pongo dia lluvioso al final

df_global.head()

Combinando datasets en df_global


,fecha,precip_mm,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,...,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso
0,1950-01-01,4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,1950-01-02,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1950-01-03,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1950-01-04,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,1950-01-05,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [27]:
total_filas = len(df_global)
conteo = df_global.notna().sum().reset_index()
conteo.columns = ["columna", "no_nulos"]
conteo["total_filas"] = total_filas
conteo["porcentaje_no_nulos"] = conteo["no_nulos"] / total_filas * 100

# Guardar como archivo de texto
with open("conteo_no_nulos.txt", "w", encoding="utf-8") as f:
    f.write(conteo.to_string(index=False))

A continuación se termina de preprocesar los datos para entrenar a los modelos (Árbol de Decisión, Naive Bayes).
Como se quiere hacer predicciones climáticas, para clasificar una instancia como "día lluvioso" o "día no lluvioso" se deben mirar los días anteriores.
Como estos modelos solo usan los atributos de una instancia para clasificar a la misma, se selecciona un "tamaño de ventana" (variable 'ventana') que corresponde a la cantidad de días previos que se considerarán para la predicción. En función de esta variable, se sustituye cada fila de atributos en los datos por una combinación (ej. promedio, máximo, etc.) de los atributos de los días anteriores.
De este modo, por ejemplo, se puede predecir si mañana lloverá, considerando los atributos hasta el día de hoy.

In [28]:
#===========COMBINACION DE DÍAS ANTERIORES PARA ENTRENAR MODELOS======================

ventana = 3

df_modelo = df_global.copy()
df_modelo = df_modelo.drop(columns=['precip_mm'])

# Columnas numéricas por tipo de estadística
cols_promedio = ['hum_carrasco', 'hum_salto', 
                 'dir_viento_carrasco_prom', 'int_viento_carrasco_prom',
                 'tprom_carrasco','tprom_salto',
                 'amplitud_carrasco','amplitud_salto',
                 'int_racha_viento_carrasco_prom','int_racha_viento_salto_prom',
                 'dir_viento_salto_prom', 'int_viento_salto_prom',
                 'vis_carrasco_prom','vis_salto_prom']

cols_maximo = ['tmax_carrasco', 'tmax_salto', 
               'int_racha_viento_carrasco_max', 'int_racha_viento_carrasco_max',
               'int_viento_salto_max', 'int_racha_viento_salto_max',
               'vis_carrasco_max','vis_salto_max']

cols_minimo = ['tmin_carrasco', 'tmin_salto',
               'vis_carrasco_min','vis_salto_min']

# Columnas one-hot
cols_fen_carrasco = [c for c in df_modelo.columns if c.startswith('fen_carrasco_')]
cols_fen_salto = [c for c in df_modelo.columns if c.startswith('fen_salto_')]

# Sustituye para las columnas en
# cols_promedio: el valor del atributo por el promedio de los dias anteriores
# cols_maximo: el valor del atributo por el maximo de los dias anteriores
# cols_minimo: el valor del atributo por el minimo de los dias anteriores
# columnas one-hot (fenomenos): el valor del atributo por la suma de los dias anteriores 
#                               (representa cant. de dias en que aparecio el fenomeno)

for col in cols_promedio:
    df_modelo[col] = df_modelo[col].rolling(window=ventana).mean().shift(1)

for col in cols_maximo:
    df_modelo[col] = df_modelo[col].rolling(window=ventana).max().shift(1)

for col in cols_minimo:
    df_modelo[col] = df_modelo[col].rolling(window=ventana).min().shift(1)

for col in cols_fen_carrasco + cols_fen_salto:
    df_modelo[col] = df_modelo[col].rolling(window=ventana).sum().shift(1)

# Drop filas que tienen NaN en todas las columnas de atributos, y las que tienen NaN en día lluvioso
cols_features = cols_promedio + cols_maximo + cols_minimo + cols_fen_carrasco + cols_fen_salto
df_modelo = df_modelo.dropna(how='all', subset=cols_features).reset_index(drop=True)
df_modelo = df_modelo.dropna(subset=['dia_lluvioso']).reset_index(drop=True)

# Drop columnas 'int_racha_viento_carrasco_prom' e 'int_racha_viento_salto_prom' por tener muy pocos datos
# df_modelo = df_modelo.drop(columns=['int_racha_viento_carrasco_prom','int_racha_viento_salto_prom'])

df_modelo

#df_modelo.to_csv('../Dataset_INUMET/data.csv', index=False)

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,...,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso
0,1970-01-04,NaN,NaN,33.8,21.0,35.4,19.8,10.000000,12.133333,26.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,1970-01-05,NaN,NaN,33.8,18.8,35.4,20.0,7.866667,12.766667,24.700000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,1970-01-06,NaN,NaN,30.5,18.8,37.2,20.0,7.366667,12.600000,23.850000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1970-01-07,NaN,NaN,30.5,18.8,37.2,20.0,7.233333,10.200000,23.283333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,1970-01-08,NaN,NaN,30.5,18.0,37.2,19.8,8.100000,9.300000,23.450000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20293,2025-07-27,NaN,NaN,16.8,4.3,18.4,7.3,7.666667,8.700000,11.000000,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20294,2025-07-28,NaN,NaN,14.5,8.4,18.0,8.5,4.700000,6.166667,11.083333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20295,2025-07-29,NaN,NaN,13.3,8.8,16.5,8.6,3.800000,5.533333,11.133333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20296,2025-07-30,NaN,NaN,15.0,2.2,17.7,3.3,6.600000,8.500000,10.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


A continuación se procede a rellenar los valores vacíos, se utilizan dos estrategias:
1. Se utiliza la mediana de todos los datos. Obs: para las columnas de temperatura, se llena con la media del mes correspondiente, de existir.
2. Se utiliza una etiqueta especial "falta" para denotar que el dato no estaba



In [29]:
#1. Mediana de todos los datos

num_cols = [col for col in df_modelo.select_dtypes(include=['float', 'int']).columns if col != 'dia_lluvioso']

df_modelo_1 = df_modelo.copy()

# Para la temperatura y humedad, llenamos con la mediana mensual si existe
df_modelo_1["mes"] = df_modelo_1["fecha"].dt.month
cols_temp = ["tmax_carrasco", "tmin_carrasco", "tmax_salto", "tmin_salto"]
cols_hum = ["hum_carrasco", "hum_salto"]
media_temp_por_mes = df_modelo_1.groupby("mes")[cols_temp].mean()
for col in cols_temp:
    df_modelo_1[col] = df_modelo_1[col].fillna(df_modelo_1["mes"].map(media_temp_por_mes[col]))
media_hum_por_mes = df_modelo_1.groupby("mes")[cols_hum].mean()
for col in cols_hum:
    df_modelo_1[col] = df_modelo_1[col].fillna(df_modelo_1["mes"].map(media_hum_por_mes[col]))

# Lleno los NaN restantes con la media de toda la columna
for col in num_cols:
    med = df_modelo_1[col].median()
    df_modelo_1[col] = df_modelo_1[col].fillna(med)  

df_modelo_1 = df_modelo_1.drop(["mes"], axis=1)
df_modelo_1

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,...,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso
0,1970-01-04,82.774359,83.496618,33.8,21.0,35.4,19.8,10.000000,12.133333,26.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1970-01-05,82.774359,83.496618,33.8,18.8,35.4,20.0,7.866667,12.766667,24.700000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1970-01-06,82.774359,83.496618,30.5,18.8,37.2,20.0,7.366667,12.600000,23.850000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1970-01-07,82.774359,83.496618,30.5,18.8,37.2,20.0,7.233333,10.200000,23.283333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,1970-01-08,82.774359,83.496618,30.5,18.0,37.2,19.8,8.100000,9.300000,23.450000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20293,2025-07-27,81.953947,83.402332,16.8,4.3,18.4,7.3,7.666667,8.700000,11.000000,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20294,2025-07-28,81.953947,83.402332,14.5,8.4,18.0,8.5,4.700000,6.166667,11.083333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20295,2025-07-29,81.953947,83.402332,13.3,8.8,16.5,8.6,3.800000,5.533333,11.133333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20296,2025-07-30,81.953947,83.402332,15.0,2.2,17.7,3.3,6.600000,8.500000,10.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
#2. Etiqueta "falta" (se usa -99)

df_modelo_2 = df_modelo.copy()
for col in num_cols:
    df_modelo_2[col] = df_modelo_2[col].fillna(-99)

df_modelo_2

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,...,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso
0,1970-01-04,-99.0,-99.0,33.8,21.0,35.4,19.8,10.000000,12.133333,26.500000,...,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,1.0
1,1970-01-05,-99.0,-99.0,33.8,18.8,35.4,20.0,7.866667,12.766667,24.700000,...,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,1.0
2,1970-01-06,-99.0,-99.0,30.5,18.8,37.2,20.0,7.366667,12.600000,23.850000,...,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0.0
3,1970-01-07,-99.0,-99.0,30.5,18.8,37.2,20.0,7.233333,10.200000,23.283333,...,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,1.0
4,1970-01-08,-99.0,-99.0,30.5,18.0,37.2,19.8,8.100000,9.300000,23.450000,...,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20293,2025-07-27,-99.0,-99.0,16.8,4.3,18.4,7.3,7.666667,8.700000,11.000000,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20294,2025-07-28,-99.0,-99.0,14.5,8.4,18.0,8.5,4.700000,6.166667,11.083333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20295,2025-07-29,-99.0,-99.0,13.3,8.8,16.5,8.6,3.800000,5.533333,11.133333,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
20296,2025-07-30,-99.0,-99.0,15.0,2.2,17.7,3.3,6.600000,8.500000,10.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


A continuación se aplican criterios de discretización para las variables continuas (necesario para entrenar a los modelos).

In [31]:
#==================DISCRETIZACIÓN DE VARIABLES CONTINUAS============================

def discretizar_temps(df_discr):
    # Helper para calcular percentiles ignorando -99
    def percentiles_columna(col, percentiles):
        valid = col[col != -99]
        return np.percentile(valid, percentiles)
    
    p10_tmaxc, p30_tmaxc, p70_tmaxc, p90_tmaxc = percentiles_columna(df_discr["tmax_carrasco"], [10, 30, 70, 90])
    p10_tminc, p30_tminc, p70_tminc, p90_tminc = percentiles_columna(df_discr["tmin_carrasco"], [10, 30, 70, 90])
    p10_tpromc, p30_tpromc, p70_tpromc, p90_tpromc = percentiles_columna(df_discr["tprom_carrasco"], [10, 30, 70, 90])
    
    p10_tmaxs, p30_tmaxs, p70_tmaxs, p90_tmaxs = percentiles_columna(df_discr["tmax_salto"], [10, 30, 70, 90])
    p10_tmins, p30_tmins, p70_tmins, p90_tmins = percentiles_columna(df_discr["tmin_salto"], [10, 30, 70, 90])
    p10_tproms, p30_tproms, p70_tproms, p90_tproms = percentiles_columna(df_discr["tprom_salto"], [10, 30, 70, 90])
    
    p25_amp_c, p50_amp_c, p85_amp_c = percentiles_columna(df_discr["amplitud_carrasco"], [25, 50, 85])
    p25_amp_s, p50_amp_s, p85_amp_s = percentiles_columna(df_discr["amplitud_salto"], [25, 50, 85])

    # Funciones de categorización
    def categorizar_tmin_tmax_tprom(temp, p10, p30, p70, p90):
        if temp == -99:
            return "falta"
        elif temp < p10:
            return "muy_frio"
        elif temp < p30:
            return "fresco"
        elif temp < p70:
            return "agradable"
        elif temp < p90:
            return "calido"
        else:
            return "muy_calido"

    def categorizar_amplitud(temp, p25, p50, p85):
        if temp == -99:
            return "falta"
        elif temp < p25:
            return "variacion_baja"
        elif temp < p50:
            return "variacion_moderada"
        elif temp < p85:
            return "variacion_alta"
        else:
            return "variacion_muy_alta"

    # Aplicar discretización
    df_discr["tmax_carrasco"] = df_discr["tmax_carrasco"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tmaxc, p30_tmaxc, p70_tmaxc, p90_tmaxc))
    df_discr["tmin_carrasco"] = df_discr["tmin_carrasco"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tminc, p30_tminc, p70_tminc, p90_tminc))
    df_discr["tprom_carrasco"] = df_discr["tprom_carrasco"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tpromc, p30_tpromc, p70_tpromc, p90_tpromc))
    df_discr["tmax_salto"] = df_discr["tmax_salto"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tmaxs, p30_tmaxs, p70_tmaxs, p90_tmaxs))
    df_discr["tmin_salto"] = df_discr["tmin_salto"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tmins, p30_tmins, p70_tmins, p90_tmins))
    df_discr["tprom_salto"] = df_discr["tprom_salto"].apply(
        lambda t: categorizar_tmin_tmax_tprom(t, p10_tproms, p30_tproms, p70_tproms, p90_tproms))
    df_discr["amplitud_carrasco"] = df_discr["amplitud_carrasco"].apply(
        lambda t: categorizar_amplitud(t, p25_amp_c, p50_amp_c, p85_amp_c))
    df_discr["amplitud_salto"] = df_discr["amplitud_salto"].apply(
        lambda t: categorizar_amplitud(t, p25_amp_s, p50_amp_s, p85_amp_s))

    return df_discr

def discretizar_restantes(df):
    df_disc = df.copy()
    
    # Función auxiliar para calcular percentiles ignorando -99
    def percentiles_col(col, pts):
        valid = col[col != -99]
        return np.percentile(valid, pts)
    
    # Función para categorizar por percentiles
    def categorizar_percentiles(val, pts, etiquetas):
        if val == -99:
            return "falta"
        for i, p in enumerate(pts):
            if val < p:
                return etiquetas[i]
        return etiquetas[-1]
    
    # Función para discretizar dirección (0-360°) en puntos cardinales
    def angulo_a_cardinal(angulo):
        if angulo == -99:
            return "falta"
        cardinales = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        index = int(((angulo + 22.5) % 360) // 45)
        return cardinales[index]
    
    # Función para discretizar columnas binarias fen_* y dia_lluvioso
    def categorizar_binario(val):
        if val == -99:
            return "falta"
        elif val == 0:
            return "no"
        elif val == 1:
            return "si"
        else:
            return "falta"  # por seguridad si hay otros valores
    
    # Columnas a ignorar: temperatura y amplitud
    temp_amp_cols = [c for c in df_disc.columns if any(t in c for t in ["tmax", "tmin", "tprom", "amplitud"])]
    
    for col in df_disc.columns:
        if col == "fecha":
            continue  # no tocar la fecha
        elif col.startswith("fen_") or col == "dia_lluvioso":
            df_disc[col] = df_disc[col].apply(categorizar_binario)
        elif col in temp_amp_cols:
            continue  # ya discretizadas
        elif col.startswith("dir_"):
            df_disc[col] = df_disc[col].apply(angulo_a_cardinal)
        else:
            # columnas continuas restantes (hum, int_viento, vis, etc.)
            pts = percentiles_col(df_disc[col], [10, 30, 70, 90])
            etiquetas = ["muy_bajo", "bajo", "medio", "alto", "muy_alto"]
            df_disc[col] = df_disc[col].apply(lambda x: categorizar_percentiles(x, pts, etiquetas))
    
    return df_disc

df_modelo_1 = discretizar_temps(df_modelo_1)
df_modelo_1 = discretizar_restantes(df_modelo_1)
df_modelo_2 = discretizar_temps(df_modelo_2)
df_modelo_2 = discretizar_restantes(df_modelo_2)
df_modelo_1

,fecha,hum_carrasco,hum_salto,tmax_carrasco,tmin_carrasco,tmax_salto,tmin_salto,amplitud_carrasco,amplitud_salto,tprom_carrasco,...,fen_salto_81,fen_salto_85,fen_salto_9,fen_salto_91,fen_salto_92,fen_salto_95,fen_salto_96,fen_salto_97,fen_salto_99,dia_lluvioso
0,1970-01-04,medio,alto,muy_calido,muy_calido,muy_calido,muy_calido,variacion_alta,variacion_moderada,muy_calido,...,no,no,no,no,no,no,no,no,no,si
1,1970-01-05,medio,alto,muy_calido,muy_calido,muy_calido,muy_calido,variacion_moderada,variacion_alta,muy_calido,...,no,no,no,no,no,no,no,no,no,si
2,1970-01-06,medio,alto,calido,muy_calido,muy_calido,muy_calido,variacion_moderada,variacion_alta,muy_calido,...,no,no,no,no,no,no,no,no,no,no
3,1970-01-07,medio,alto,calido,muy_calido,muy_calido,muy_calido,variacion_moderada,variacion_moderada,muy_calido,...,no,no,no,no,no,no,no,no,no,si
4,1970-01-08,medio,alto,calido,muy_calido,muy_calido,muy_calido,variacion_moderada,variacion_baja,muy_calido,...,no,no,no,no,no,no,no,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20293,2025-07-27,bajo,alto,fresco,fresco,muy_frio,agradable,variacion_moderada,variacion_baja,fresco,...,no,no,no,si,si,si,no,no,no,no
20294,2025-07-28,bajo,alto,muy_frio,agradable,muy_frio,agradable,variacion_baja,variacion_baja,fresco,...,no,no,no,si,si,si,no,no,no,no
20295,2025-07-29,bajo,alto,muy_frio,agradable,muy_frio,agradable,variacion_baja,variacion_baja,fresco,...,no,no,no,si,si,si,no,no,no,no
20296,2025-07-30,bajo,alto,muy_frio,muy_frio,muy_frio,fresco,variacion_baja,variacion_baja,fresco,...,no,no,no,no,no,no,no,no,no,no


In [32]:
from astral import moon
import pandas as pd

def calcular_fases_lunares(df, col_fecha='fecha'):
    df = df.copy()
    df[col_fecha] = pd.to_datetime(df[col_fecha])
    
    def fase_lunar(fecha):
        dia = moon.phase(fecha)  # día del ciclo lunar
        # Clasificación simplificada en 4 fases
        if dia < 1 or dia > 28:
            return "luna_nueva"
        elif dia < 7:
            return "luna_creciente"
        elif dia < 15:
            return "luna_llena"
        elif dia < 22:
            return "luna_menguante"
        else:
            return "luna_nueva"
    
    df['fase_lunar'] = df[col_fecha].apply(fase_lunar)
    return df

df_modelo_1 = calcular_fases_lunares(df_modelo_1)
df_modelo_2 = calcular_fases_lunares(df_modelo_2)
df_modelo_1.to_csv('../Dataset_INUMET/data.csv', index=False)

In [33]:
#df_modelo_1.to_excel("df_modelo_1_discretizado.xlsx", index=False)
#df_modelo_2.to_excel("df_modelo_2_discretizado.xlsx", index=False)
#import sys
#print(sys.executable)